# paper-read — PDF → Markdown(LaTeX) 변환 (무료 GPU)

Google Colab의 무료 **T4 GPU**로 Marker OCR를 돌려 논문 PDF를 선택가능 텍스트 + LaTeX 수식 마크다운으로 변환합니다.

**쓰는 법**
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 구글 드라이브 `MyDrive/paper-read/sources/` 폴더에 PDF를 올림
3. 아래 셀을 위에서부터 순서대로 실행 (▶️)
4. 결과는 `MyDrive/paper-read/marker_out/` 에 저장됨 → 로컬로 내려받아 `python3 scripts/marker_to_json.py marker_out/<slug> <slug>` 실행


In [ ]:
# 1) GPU 확인 (T4가 보여야 함)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2) 구글 드라이브 연결 (팝업 인증)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Marker 설치 (처음 1회, ~1–2분)
!pip -q install marker-pdf

In [ ]:
# 4) 경로 설정 + 모델 캐시를 Drive에 두어 다음 실행부터 재다운로드 없게
import os
BASE = '/content/drive/MyDrive/paper-read'
IN, OUT = f'{BASE}/sources', f'{BASE}/marker_out'
os.makedirs(IN, exist_ok=True); os.makedirs(OUT, exist_ok=True)
# Surya 모델(~3.3GB)을 Drive에 캐시 → 세션 다시 열어도 재다운로드 X
os.environ['HF_HOME'] = f'{BASE}/.cache/huggingface'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
pdfs = [f for f in os.listdir(IN) if f.lower().endswith('.pdf')]
print(f'PDFs to convert ({len(pdfs)}):'); print('\n'.join(pdfs) or '  (none — upload to '+IN+')')

In [ ]:
# 5) 변환 실행 (T4 16GB → workers=1 권장). 4편에 수 분.
!TORCH_DEVICE=cuda HF_HOME="$BASE/.cache/huggingface" \
  marker "{IN}" --output_dir "{OUT}" --output_format markdown --workers 1
print('\n✅ done. results in', OUT)

In [ ]:
# 6) (선택) 결과를 zip으로 묶어 바로 내려받기
import shutil
shutil.make_archive('/content/marker_out', 'zip', OUT)
from google.colab import files
files.download('/content/marker_out.zip')

## 다음 단계 (로컬)

내려받은 `marker_out/`을 레포 루트에 두고, 논문별로:

```bash
python3 scripts/marker_to_json.py marker_out/<slug> <slug>
npm run dev   # 미리보기
git add -A && git commit -m 'add papers' && git push   # Vercel 자동 배포
```
